In [2]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql import types as T
from pyspark.sql import Row
from pyspark.sql import Window

spark = SparkSession \
    .builder \
    .master("local") \
    .config("spark.driver.memory", "4g") \
    .appName("ex5_google_review_ingestion") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/11 21:32:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
reviews_df = spark.read.csv('s3a://pyspark/data/raw/google_reviews/', header=True)

26/08/11 21:32:35 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [34]:
sentiment_rank_arr =  [Row(Sentiment='Positive', assign=1),
                       Row(Sentiment='Neutral', assign=0),
                       Row(Sentiment='Negative', assign=-1)                      
]

sentiment_rank_df = spark.createDataFrame(sentiment_rank_arr)

In [35]:
joined_df = reviews_df.join(F.broadcast(sentiment_rank_df), on='Sentiment', how='left')

joined_df.show(5)

+--------------------+--------------------+--------------------+------------------+----------------------+------+
|           Sentiment|                 App|   Translated_Review|Sentiment_Polarity|Sentiment_Subjectivity|assign|
+--------------------+--------------------+--------------------+------------------+----------------------+------+
| also ""Best Befo...|10 Best Foods for...|"I like eat delic...|          Positive|                   1.0|  null|
|            Positive|10 Best Foods for...|This help eating ...|              0.25|   0.28846153846153844|     1|
|                 nan|10 Best Foods for...|                 nan|               nan|                   nan|  null|
|            Positive|10 Best Foods for...|Works great espec...|               0.4|                 0.875|     1|
|            Positive|10 Best Foods for...|        Best idea us|               1.0|                   0.3|     1|
+--------------------+--------------------+--------------------+------------------+-----

In [36]:
selected_df = joined_df\
.select(F.col('App').alias('application_name'),
        F.col('Translated_review').alias('translated_review'),
        F.col('assign').cast(T.LongType()).alias('sentiment_rank'),
        F.col('Sentiment_Polarity').cast(T.FloatType()).alias('sentiment_polarity'),
        F.col('Sentiment_Subjectivity').cast(T.FloatType()).alias('sentiment_subjectivity')
)

selected_df.show(5)

selected_df.write.parquet('s3a://pyspark/data/source/google_reviews/', mode='overwrite')

+--------------------+--------------------+--------------+------------------+----------------------+
|    application_name|   translated_review|sentiment_rank|sentiment_polarity|sentiment_subjectivity|
+--------------------+--------------------+--------------+------------------+----------------------+
|10 Best Foods for...|"I like eat delic...|          null|              null|                   1.0|
|10 Best Foods for...|This help eating ...|             1|              0.25|            0.28846154|
|10 Best Foods for...|                 nan|          null|               NaN|                   NaN|
|10 Best Foods for...|Works great espec...|             1|               0.4|                 0.875|
|10 Best Foods for...|        Best idea us|             1|               1.0|                   0.3|
+--------------------+--------------------+--------------+------------------+----------------------+
only showing top 5 rows

